#  Actividad Computacional 2: Álgebra Lineal y Modelos de Lenguaje

## Construyendo *MiniGPT*: El Álgebra Lineal detrás de un Modelo de Lenguaje

---

### 📋 Instrucciones Generales

- **Duración:** 2 horas.
- **Objetivo:** Evaluar su comprensión de **bases**, **sistemas de ecuaciones lineales**, **tipos de solución**, **operaciones elementales** y **reducción de Gauss**, en el contexto de un modelo de lenguaje simplificado.
- **Cómo trabajar:**
  1. Ejecute cada celda de código en orden (botón ▶ o `Shift+Enter`).
  2. Interactúe con los *sliders*, menús desplegables y botones que aparecen.
  3. **Escriba TODAS sus respuestas** en las cajas de texto provistas dentro del cuaderno.
  4. **No necesita programar.** Solo ejecute las celdas y use los controles interactivos.
  5. Al finalizar, guarde el cuaderno (`Ctrl+S` / `Cmd+S`) y súbalo a la plataforma.

###  Contexto

Los modelos de lenguaje como GPT procesan texto mediante operaciones de álgebra lineal:
- Cada **palabra** se representa como un **vector numérico** llamado *embedding*.
- El **entrenamiento** del modelo implica resolver **sistemas de ecuaciones lineales** para encontrar los pesos óptimos.
- La **reducción de Gauss** y las **operaciones elementales** son herramientas fundamentales en estos procesos.

En esta actividad, usted asumirá el rol de ingeniero(a) construyendo **MiniGPT**, un modelo de lenguaje simplificado. En cada parte enfrentará desafíos reales que requieren comprender estos conceptos.

---

In [ ]:
# ═══════════════════════════════════════════════════════════════
# EJECUTE ESTA CELDA PRIMERO (configuración inicial)
# ═══════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Markdown
import warnings
warnings.filterwarnings('ignore')

# Diccionario global para almacenar respuestas
respuestas = {}

def caja_respuesta(pregunta_id, texto_pregunta, filas=3):
    # Crea una caja de texto para que el estudiante responda.
    label = widgets.HTML(
        value=f'<div style="background:#1a1a2e; color:#e0e0ff; padding:10px; '
              f'border-left:4px solid #4a90d9; margin:5px 0; border-radius:4px;">'
              f'<b>📝 {pregunta_id}:</b> {texto_pregunta}</div>'
    )
    textarea = widgets.Textarea(
        value='',
        placeholder='Escriba su respuesta aquí...',
        layout=widgets.Layout(width='95%', height=f'{filas * 28}px')
    )
    def guardar(change):
        respuestas[pregunta_id] = change['new']
    textarea.observe(guardar, names='value')
    display(label, textarea)

print("✅ Configuración completada exitosamente.")
print("Puede continuar ejecutando las celdas siguientes.")

---
## Parte 1: Embeddings de Palabras y el Concepto de Base (~30 min)

### Contexto

Para que MiniGPT entienda palabras, cada una se convierte en un **vector numérico** (embedding). Por ejemplo, si usamos un espacio de dimensión 2 ($\mathbb{R}^2$), cada palabra sería un punto en el plano:

| Palabra | Embedding |
|---------|-----------|
| "gato"  | $(2, 1)$  |
| "perro" | $(1, 3)$  |
| "casa"  | $(3, 4)$  |

La pregunta clave es: **¿los vectores de embedding son suficientes para representar cualquier significado posible?** Esto nos lleva al concepto de **base**.

---

### Actividad 1.1: Explorando embeddings en el plano

Ejecute la celda siguiente. Aparecerán deslizadores (*sliders*) que controlan las componentes de dos vectores de embedding $\mathbf{e}_1$ y $\mathbf{e}_2$ en $\mathbb{R}^2$. Mueva los *sliders* y observe cuándo estos vectores forman una **base** del plano.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ACTIVIDAD 1.1 - Ejecute esta celda y use los sliders
# ═══════════════════════════════════════════════════════════════

output_11 = widgets.Output()

def plot_vectors_2d(e1x, e1y, e2x, e2y):
    with output_11:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

        # --- Panel izquierdo: vectores ---
        ax = axes[0]
        ax.set_xlim(-5.5, 5.5)
        ax.set_ylim(-5.5, 5.5)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='k', linewidth=0.5)
        ax.axvline(x=0, color='k', linewidth=0.5)

        ax.quiver(0, 0, e1x, e1y, angles='xy', scale_units='xy', scale=1,
                   color='#2196F3', linewidth=2.5, label=f'e₁ = ({e1x:.1f}, {e1y:.1f})')
        ax.quiver(0, 0, e2x, e2y, angles='xy', scale_units='xy', scale=1,
                   color='#F44336', linewidth=2.5, label=f'e₂ = ({e2x:.1f}, {e2y:.1f})')

        det = e1x * e2y - e1y * e2x

        if abs(det) < 0.05:
            ax.set_title('⚠️ Vectores LINEALMENTE DEPENDIENTES\nNO forman base de R²',
                         color='red', fontsize=12, fontweight='bold')
            if abs(e1x) + abs(e1y) > 0.01:
                t_vals = np.linspace(-6, 6, 100)
                ax.plot(e1x * t_vals, e1y * t_vals, 'purple', alpha=0.3, linewidth=8,
                        label='Espacio generado (solo una recta)')
        else:
            ax.set_title('✅ Vectores LINEALMENTE INDEPENDIENTES\nForman BASE de R²',
                         color='green', fontsize=12, fontweight='bold')

        ax.legend(loc='upper left', fontsize=9)
        ax.set_xlabel('Componente x')
        ax.set_ylabel('Componente y')

        # --- Panel derecho: info ---
        ax2 = axes[1]
        ax2.axis('off')
        info = f"Información del sistema de embeddings\n"
        info += f"{'='*42}\n\n"
        info += f"Vector e₁ = ({e1x:.1f}, {e1y:.1f})\n"
        info += f"Vector e₂ = ({e2x:.1f}, {e2y:.1f})\n\n"
        info += f"Determinante de [e₁ | e₂] = {det:.2f}\n\n"

        if abs(det) < 0.05:
            info += "Estado: LINEALMENTE DEPENDIENTES\n"
            info += "→ No forman base de R²\n"
            info += "→ Solo generan una recta (o el origen)\n"
            info += "→ MiniGPT NO podría distinguir\n"
            info += "  todos los significados posibles"
        else:
            info += "Estado: LINEALMENTE INDEPENDIENTES\n"
            info += "→ Forman base de R²\n"
            info += "→ Generan todo R²\n"
            info += "→ MiniGPT puede representar\n"
            info += "  cualquier significado en el plano"

        ax2.text(0.05, 0.95, info, transform=ax2.transAxes, fontsize=11,
                 verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

        plt.tight_layout()
        plt.show()

s_e1x = widgets.FloatSlider(value=2, min=-5, max=5, step=0.1, description='e₁ₓ:',
                              style={'description_width': '40px'}, layout=widgets.Layout(width='45%'))
s_e1y = widgets.FloatSlider(value=1, min=-5, max=5, step=0.1, description='e₁ᵧ:',
                              style={'description_width': '40px'}, layout=widgets.Layout(width='45%'))
s_e2x = widgets.FloatSlider(value=1, min=-5, max=5, step=0.1, description='e₂ₓ:',
                              style={'description_width': '40px'}, layout=widgets.Layout(width='45%'))
s_e2y = widgets.FloatSlider(value=3, min=-5, max=5, step=0.1, description='e₂ᵧ:',
                              style={'description_width': '40px'}, layout=widgets.Layout(width='45%'))

ui = widgets.VBox([
    widgets.HTML('<h4>🎛️ Ajuste las componentes de los vectores de embedding:</h4>'),
    widgets.HBox([s_e1x, s_e1y]),
    widgets.HBox([s_e2x, s_e2y]),
])

out = widgets.interactive_output(plot_vectors_2d, {'e1x': s_e1x, 'e1y': s_e1y, 'e2x': s_e2x, 'e2y': s_e2y})
display(ui, output_11)
plot_vectors_2d(s_e1x.value, s_e1y.value, s_e2x.value, s_e2y.value)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PREGUNTAS 1.1 — Escriba sus respuestas abajo
# ═══════════════════════════════════════════════════════════════

caja_respuesta("P1",
    "Ajuste los sliders para que e₁ y e₂ sean linealmente dependientes. "
    "Describa las coordenadas que usó y lo que observa geométricamente. "
    "¿Qué relación matemática existe entre los dos vectores en esa configuración?", 4)

caja_respuesta("P2",
    "Cuando e₁ y e₂ son linealmente independientes, ¿forman una base de R²? "
    "Justifique usando la definición de base (independencia lineal + generación del espacio).", 4)

caja_respuesta("P3",
    "Si el espacio de embeddings de MiniGPT es R², ¿cuántos vectores necesita exactamente una base? "
    "¿Podría tener una base con 3 vectores? ¿Y con 1 vector? Explique por qué.", 4)

### Actividad 1.2: Combinaciones lineales y representación de palabras

Si $\{\mathbf{e}_1, \mathbf{e}_2\}$ es una base, entonces **cualquier** vector en $\mathbb{R}^2$ se puede escribir de manera **única** como $\alpha \mathbf{e}_1 + \beta \mathbf{e}_2$.

En la siguiente celda, se muestran dos vectores base fijos y un vector **objetivo** (la palabra "casa" que queremos representar). Use los *sliders* $\alpha$ y $\beta$ para encontrar la combinación lineal que alcanza el objetivo.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ACTIVIDAD 1.2 - Combinaciones lineales
# ═══════════════════════════════════════════════════════════════

output_12 = widgets.Output()

e1_fijo = np.array([2, 1])
e2_fijo = np.array([1, 3])
objetivo = np.array([3, 4])

def plot_combinacion(alpha, beta):
    with output_12:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.set_xlim(-8, 8)
        ax.set_ylim(-8, 8)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='k', linewidth=0.5)
        ax.axvline(x=0, color='k', linewidth=0.5)

        combo = alpha * e1_fijo + beta * e2_fijo

        ax.quiver(0, 0, e1_fijo[0], e1_fijo[1], angles='xy', scale_units='xy', scale=1,
                   color='#2196F3', linewidth=2, alpha=0.5, label=f'e₁ = ({e1_fijo[0]}, {e1_fijo[1]})')
        ax.quiver(0, 0, e2_fijo[0], e2_fijo[1], angles='xy', scale_units='xy', scale=1,
                   color='#F44336', linewidth=2, alpha=0.5, label=f'e₂ = ({e2_fijo[0]}, {e2_fijo[1]})')

        ax.quiver(0, 0, alpha*e1_fijo[0], alpha*e1_fijo[1], angles='xy', scale_units='xy', scale=1,
                   color='#2196F3', linewidth=1.5, linestyle='dashed')
        ax.quiver(alpha*e1_fijo[0], alpha*e1_fijo[1], beta*e2_fijo[0], beta*e2_fijo[1],
                   angles='xy', scale_units='xy', scale=1,
                   color='#F44336', linewidth=1.5, linestyle='dashed')

        ax.quiver(0, 0, combo[0], combo[1], angles='xy', scale_units='xy', scale=1,
                   color='#4CAF50', linewidth=3, label=f'αe₁+βe₂ = ({combo[0]:.1f}, {combo[1]:.1f})')

        ax.plot(objetivo[0], objetivo[1], 'k*', markersize=20, label=f'Objetivo "casa" = ({objetivo[0]}, {objetivo[1]})')

        dist = np.linalg.norm(combo - objetivo)
        if dist < 0.15:
            ax.set_title('🎯 ¡OBJETIVO ALCANZADO! La palabra "casa" está representada.',
                         fontsize=12, color='green', fontweight='bold')
        else:
            ax.set_title(f'Distancia al objetivo: {dist:.2f} — Siga ajustando α y β',
                         fontsize=12, color='orange')

        ax.legend(loc='upper left', fontsize=9)
        ax.set_xlabel('Componente x')
        ax.set_ylabel('Componente y')
        plt.tight_layout()
        plt.show()

s_alpha = widgets.FloatSlider(value=0, min=-5, max=5, step=0.1, description='α:',
                               style={'description_width': '30px'}, layout=widgets.Layout(width='70%'))
s_beta = widgets.FloatSlider(value=0, min=-5, max=5, step=0.1, description='β:',
                              style={'description_width': '30px'}, layout=widgets.Layout(width='70%'))

ui2 = widgets.VBox([
    widgets.HTML('<h4>🎛️ Ajuste α y β para representar la palabra "casa" = (3, 4):</h4>'),
    s_alpha, s_beta
])

out2 = widgets.interactive_output(plot_combinacion, {'alpha': s_alpha, 'beta': s_beta})
display(ui2, output_12)
plot_combinacion(s_alpha.value, s_beta.value)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PREGUNTAS 1.2 — Escriba sus respuestas abajo
# ═══════════════════════════════════════════════════════════════

caja_respuesta("P4",
    "¿Qué valores de α y β encontró para representar 'casa' = (3, 4) como combinación "
    "lineal de e₁ = (2,1) y e₂ = (1,3)? Plantee y resuelva el sistema de ecuaciones correspondiente "
    "para verificar su respuesta.", 4)

caja_respuesta("P5",
    "Si los vectores e₁ y e₂ fueran linealmente dependientes (por ejemplo, e₂ = 2·e₁), "
    "¿podría representar cualquier palabra del plano? ¿Qué implicación tiene esto para la capacidad "
    "de MiniGPT de distinguir significados?", 4)

caja_respuesta("P6",
    "Si MiniGPT usa un espacio de embeddings de dimensión 3 (R³), ¿cuántos vectores "
    "linealmente independientes necesitaría como mínimo para que cualquier palabra "
    "tenga una representación única? Justifique su respuesta usando el concepto de base.", 3)

---
## Parte 2: Operaciones Elementales y Reducción de Gauss (~35 min)

### Contexto

Para encontrar los **pesos** de MiniGPT, necesitamos resolver sistemas de ecuaciones lineales. La herramienta principal para esto es la **reducción de Gauss**, que usa tres **operaciones elementales por fila**:

1. **Intercambiar** dos filas: $F_i \leftrightarrow F_j$
2. **Multiplicar** una fila por un escalar no nulo: $F_i \to k \cdot F_i$
3. **Sumar** un múltiplo de una fila a otra: $F_i \to F_i + k \cdot F_j$

Estas operaciones **no cambian** el conjunto solución del sistema.

---

### Actividad 2.1: Reducción de Gauss interactiva

Suponga que durante el entrenamiento de MiniGPT, se obtiene el siguiente sistema de ecuaciones para calcular tres pesos $x_1, x_2, x_3$:

$$\begin{cases} 2x_1 + 4x_2 - 2x_3 = 2 \\ x_1 + 2x_2 + x_3 = 4 \\ 3x_1 + 2x_2 + x_3 = 7 \end{cases}$$

Use el panel interactivo para aplicar operaciones elementales y llevar la **matriz aumentada** a su **forma escalonada**.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ACTIVIDAD 2.1 - Reducción de Gauss interactiva
# ═══════════════════════════════════════════════════════════════

class GaussInteractivo:
    def __init__(self, matriz, nombres_vars=['x₁', 'x₂', 'x₃']):
        self.original = np.array(matriz, dtype=float)
        self.M = self.original.copy()
        self.nombres = nombres_vars
        self.historial = ['Matriz inicial']
        self.matrices = [self.original.copy()]
        self.n_filas = self.M.shape[0]
        self.output = widgets.Output()

        self.op_tipo = widgets.Dropdown(
            options=['Intercambiar filas (Fi ↔ Fj)',
                     'Multiplicar fila (Fi → k·Fi)',
                     'Sumar múltiplo (Fi → Fi + k·Fj)'],
            description='Operación:', style={'description_width': '80px'},
            layout=widgets.Layout(width='60%'))

        self.fila_i = widgets.Dropdown(options=[(f'Fila {i+1}', i) for i in range(self.n_filas)],
                                        description='Fila i:', style={'description_width': '80px'})
        self.fila_j = widgets.Dropdown(options=[(f'Fila {j+1}', j) for j in range(self.n_filas)],
                                        description='Fila j:', value=1,
                                        style={'description_width': '80px'})
        self.escalar = widgets.FloatText(value=1.0, description='k =',
                                          style={'description_width': '30px'},
                                          layout=widgets.Layout(width='150px'))

        self.btn_aplicar = widgets.Button(description='▶ Aplicar operación',
                                           button_style='primary',
                                           layout=widgets.Layout(width='200px'))
        self.btn_deshacer = widgets.Button(description='↩ Deshacer último paso',
                                            button_style='warning',
                                            layout=widgets.Layout(width='200px'))
        self.btn_reiniciar = widgets.Button(description='🔄 Reiniciar',
                                             button_style='danger',
                                             layout=widgets.Layout(width='200px'))

        self.btn_aplicar.on_click(self._aplicar)
        self.btn_deshacer.on_click(self._deshacer)
        self.btn_reiniciar.on_click(self._reiniciar)

        controles = widgets.VBox([
            widgets.HTML('<h4>🔧 Panel de Operaciones Elementales</h4>'),
            self.op_tipo,
            widgets.HBox([self.fila_i, self.fila_j, self.escalar]),
            widgets.HBox([self.btn_aplicar, self.btn_deshacer, self.btn_reiniciar])
        ])

        display(controles, self.output)
        self._mostrar()

    def _formato_matriz(self):
        # Formato HTML de la matriz aumentada
        n = self.M.shape[1] - 1
        html = '<table style="border-collapse:collapse; font-size:16px; margin:10px auto;">'
        for i in range(self.M.shape[0]):
            html += '<tr>'
            for j in range(self.M.shape[1]):
                val = self.M[i, j]
                s = f'{val:.2f}' if val != int(val) else f'{int(val)}'
                border = 'border-left:3px solid black; padding-left:10px;' if j == n else ''
                bg = '#e8f5e9' if j == n else '#ffffff'
                html += f'<td style="padding:6px 12px; text-align:center; {border} background:{bg};">{s}</td>'
            html += '</tr>'
        html += '</table>'
        return html

    def _mostrar(self):
        with self.output:
            clear_output(wait=True)
            display(HTML(self._formato_matriz()))
            if len(self.historial) > 1:
                pasos = '<br>'.join([f'<b>Paso {i}:</b> {h}' for i, h in enumerate(self.historial) if i > 0])
                display(HTML(f'<div style="background:#f0f0f0; padding:10px; border-radius:5px;">'
                              f'<b>📋 Historial de operaciones:</b><br>{pasos}</div>'))

    def _aplicar(self, _):
        i = self.fila_i.value
        j = self.fila_j.value
        k = self.escalar.value
        op = self.op_tipo.value

        if 'Intercambiar' in op:
            if i == j:
                with self.output:
                    clear_output(wait=True)
                    display(HTML('<p style="color:red;">⚠️ Seleccione dos filas diferentes.</p>'))
                    display(HTML(self._formato_matriz()))
                return
            self.M[[i, j]] = self.M[[j, i]]
            desc = f'F{i+1} ↔ F{j+1}'
        elif 'Multiplicar' in op:
            if abs(k) < 1e-10:
                with self.output:
                    clear_output(wait=True)
                    display(HTML('<p style="color:red;">⚠️ El escalar no puede ser cero.</p>'))
                    display(HTML(self._formato_matriz()))
                return
            self.M[i] = k * self.M[i]
            desc = f'F{i+1} → {k}·F{i+1}'
        else:
            self.M[i] = self.M[i] + k * self.M[j]
            desc = f'F{i+1} → F{i+1} + ({k})·F{j+1}'

        self.historial.append(desc)
        self.matrices.append(self.M.copy())
        self._mostrar()

    def _deshacer(self, _):
        if len(self.matrices) > 1:
            self.matrices.pop()
            self.historial.pop()
            self.M = self.matrices[-1].copy()
            self._mostrar()

    def _reiniciar(self, _):
        self.M = self.original.copy()
        self.historial = ['Matriz inicial']
        self.matrices = [self.original.copy()]
        self._mostrar()

# Sistema 1: solución única
print("Sistema de ecuaciones para los pesos de MiniGPT:")
print("  2x₁ + 4x₂ − 2x₃ = 2")
print("   x₁ + 2x₂ +  x₃ = 4")
print("  3x₁ + 2x₂ +  x₃ = 7")
print()
gauss1 = GaussInteractivo([[2, 4, -2, 2],
                             [1, 2, 1, 4],
                             [3, 2, 1, 7]])

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PREGUNTAS 2.1 — Escriba sus respuestas abajo
# ═══════════════════════════════════════════════════════════════

caja_respuesta("P7",
    "Liste en orden las operaciones elementales que aplicó para llevar la matriz a forma escalonada. "
    "(Ejemplo: Paso 1: F1 ↔ F2, Paso 2: F3 → F3 + (-3)·F1, etc.)", 5)

caja_respuesta("P8",
    "A partir de la forma escalonada que obtuvo, ¿cuántos pivotes tiene la matriz? "
    "¿Cuántas soluciones tiene el sistema? Si es solución única, indique los valores de x₁, x₂, x₃.", 4)

caja_respuesta("P9",
    "¿Las operaciones elementales que aplicó cambiaron el conjunto solución del sistema? "
    "Justifique su respuesta.", 3)

### Actividad 2.2: Otro sistema de entrenamiento

Ahora suponga que los datos de entrenamiento de MiniGPT generan este sistema:

$$\begin{cases} x_1 + 2x_2 + 3x_3 = 6 \\ 2x_1 + 4x_2 + 6x_3 = 12 \\ x_1 + 3x_2 + 5x_3 = 10 \end{cases}$$

Aplique la reducción de Gauss y analice qué sucede.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ACTIVIDAD 2.2 - Segundo sistema
# ═══════════════════════════════════════════════════════════════

print("Segundo sistema de ecuaciones:")
print("   x₁ + 2x₂ + 3x₃ = 6")
print("  2x₁ + 4x₂ + 6x₃ = 12")
print("   x₁ + 3x₂ + 5x₃ = 10")
print()
gauss2 = GaussInteractivo([[1, 2, 3, 6],
                             [2, 4, 6, 12],
                             [1, 3, 5, 10]])

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PREGUNTAS 2.2 — Escriba sus respuestas abajo
# ═══════════════════════════════════════════════════════════════

caja_respuesta("P10",
    "Después de reducir la segunda matriz, ¿obtuvo alguna fila de ceros? "
    "¿Cuántos pivotes tiene? ¿Cuántas variables libres hay?", 4)

caja_respuesta("P11",
    "¿Cuántas soluciones tiene el segundo sistema: ninguna, una única, o infinitas? "
    "Explique cómo lo determina a partir de la forma escalonada y "
    "escriba la solución general si corresponde.", 5)

caja_respuesta("P12",
    "Comparando ambos sistemas: ¿cuál es la diferencia fundamental entre tener "
    "tantos pivotes como variables (sistema 1) versus tener menos pivotes que variables (sistema 2)? "
    "¿Qué implicación tiene esto para los pesos de MiniGPT?", 4)

---
## Parte 3: Sistemas de Ecuaciones y Tipos de Solución (~30 min)

### Contexto

Cuando MiniGPT se entrena con diferentes conjuntos de datos, los sistemas de ecuaciones resultantes pueden tener:
- **Solución única**: El modelo tiene pesos perfectamente determinados (caso ideal).
- **Infinitas soluciones**: El modelo está *subdeterminado*; hay múltiples configuraciones de pesos posibles (se necesita regularización).
- **Sin solución**: Los datos de entrenamiento son *inconsistentes*; el modelo no puede ajustarse perfectamente.

Exploremos cómo identificar cada caso tanto de forma geométrica (en 2D) como algebraica.

---

### Actividad 3.1: Geometría de sistemas 2×2

Un sistema de 2 ecuaciones con 2 incógnitas corresponde geométricamente a **dos rectas** en el plano. Use los *sliders* para modificar los coeficientes y observe cómo cambia la solución.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ACTIVIDAD 3.1 - Visualización geométrica de sistemas 2×2
# ═══════════════════════════════════════════════════════════════

output_31 = widgets.Output()

def plot_sistema_2x2(a1, b1, c1, a2, b2, c2):
    with output_31:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 6.5))
        x = np.linspace(-10, 10, 1000)

        lineas_dibujadas = False

        if abs(b1) > 0.001:
            y1 = (c1 - a1 * x) / b1
            mask1 = np.abs(y1) < 12
            ax.plot(x[mask1], y1[mask1], 'b-', linewidth=2.5,
                    label=f'Ec.1: {a1:.0f}x + {b1:.0f}y = {c1:.0f}')
            lineas_dibujadas = True
        elif abs(a1) > 0.001:
            ax.axvline(x=c1/a1, color='b', linewidth=2.5,
                       label=f'Ec.1: {a1:.0f}x = {c1:.0f}')
            lineas_dibujadas = True

        if abs(b2) > 0.001:
            y2 = (c2 - a2 * x) / b2
            mask2 = np.abs(y2) < 12
            ax.plot(x[mask2], y2[mask2], 'r-', linewidth=2.5,
                    label=f'Ec.2: {a2:.0f}x + {b2:.0f}y = {c2:.0f}')
            lineas_dibujadas = True
        elif abs(a2) > 0.001:
            ax.axvline(x=c2/a2, color='r', linewidth=2.5,
                       label=f'Ec.2: {a2:.0f}x = {c2:.0f}')
            lineas_dibujadas = True

        det = a1 * b2 - a2 * b1

        if abs(det) > 0.001:
            sol_x = (c1*b2 - c2*b1) / det
            sol_y = (a1*c2 - a2*c1) / det
            ax.plot(sol_x, sol_y, 'go', markersize=15, zorder=5, markeredgecolor='black',
                    markeredgewidth=2, label=f'Solución: ({sol_x:.2f}, {sol_y:.2f})')
            titulo = f'✅ SOLUCIÓN ÚNICA: ({sol_x:.2f}, {sol_y:.2f})'
            color_t = 'green'
            tipo = 'Solución única'
        else:
            prop1 = a1 * c2 - a2 * c1 if abs(a1) + abs(a2) > 0 else b1 * c2 - b2 * c1
            if abs(prop1) < 0.001:
                titulo = '🔵 INFINITAS SOLUCIONES (rectas coincidentes)'
                color_t = 'blue'
                tipo = 'Infinitas soluciones'
            else:
                titulo = '🔴 SIN SOLUCIÓN (rectas paralelas)'
                color_t = 'red'
                tipo = 'Sin solución'

        ax.set_xlim(-10, 10)
        ax.set_ylim(-10, 10)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='k', linewidth=0.5)
        ax.axvline(x=0, color='k', linewidth=0.5)
        ax.set_xlabel('x', fontsize=12)
        ax.set_ylabel('y', fontsize=12)
        ax.set_title(titulo, fontsize=13, color=color_t, fontweight='bold')
        ax.legend(fontsize=10, loc='upper right')

        info_text = f"Tipo: {tipo}\ndet = {a1:.0f}·{b2:.0f} - {a2:.0f}·{b1:.0f} = {det:.1f}"
        ax.text(0.02, 0.02, info_text, transform=ax.transAxes, fontsize=10,
                verticalalignment='bottom', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

        plt.tight_layout()
        plt.show()

sa1 = widgets.IntSlider(value=2, min=-5, max=5, description='a₁:', style={'description_width':'30px'}, layout=widgets.Layout(width='30%'))
sb1 = widgets.IntSlider(value=1, min=-5, max=5, description='b₁:', style={'description_width':'30px'}, layout=widgets.Layout(width='30%'))
sc1 = widgets.IntSlider(value=5, min=-10, max=10, description='c₁:', style={'description_width':'30px'}, layout=widgets.Layout(width='30%'))
sa2 = widgets.IntSlider(value=1, min=-5, max=5, description='a₂:', style={'description_width':'30px'}, layout=widgets.Layout(width='30%'))
sb2 = widgets.IntSlider(value=-1, min=-5, max=5, description='b₂:', style={'description_width':'30px'}, layout=widgets.Layout(width='30%'))
sc2 = widgets.IntSlider(value=1, min=-10, max=10, description='c₂:', style={'description_width':'30px'}, layout=widgets.Layout(width='30%'))

ui3 = widgets.VBox([
    widgets.HTML('<h4>🎛️ Ecuación 1: a₁·x + b₁·y = c₁</h4>'),
    widgets.HBox([sa1, sb1, sc1]),
    widgets.HTML('<h4>🎛️ Ecuación 2: a₂·x + b₂·y = c₂</h4>'),
    widgets.HBox([sa2, sb2, sc2]),
])

out3 = widgets.interactive_output(plot_sistema_2x2, {'a1':sa1,'b1':sb1,'c1':sc1,'a2':sa2,'b2':sb2,'c2':sc2})
display(ui3, output_31)
plot_sistema_2x2(sa1.value, sb1.value, sc1.value, sa2.value, sb2.value, sc2.value)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PREGUNTAS 3.1 — Escriba sus respuestas abajo
# ═══════════════════════════════════════════════════════════════

caja_respuesta("P13",
    "Ajuste los sliders para obtener un sistema con SOLUCIÓN ÚNICA. Escriba las dos "
    "ecuaciones que usó y las coordenadas de la solución. ¿Qué condición cumple el determinante?", 4)

caja_respuesta("P14",
    "Ahora ajuste para obtener un sistema SIN SOLUCIÓN. Escriba las ecuaciones. "
    "¿Qué observa geométricamente? ¿Qué relación hay entre los coeficientes de ambas ecuaciones?", 4)

caja_respuesta("P15",
    "Finalmente, ajuste para obtener INFINITAS SOLUCIONES. Escriba las ecuaciones. "
    "¿Qué relación existe entre las dos ecuaciones? ¿Qué valor tiene el determinante?", 4)

### Actividad 3.2: Sistema paramétrico en el entrenamiento

Suponga que los datos de entrenamiento de MiniGPT dependen de un hiperparámetro $\lambda$ (tasa de aprendizaje). El sistema resultante es:

$$\begin{cases} x_1 + 2x_2 = 3 \\ 2x_1 + \lambda x_2 = 6 \end{cases}$$

Explore cómo el valor de $\lambda$ afecta el **tipo de solución**.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ACTIVIDAD 3.2 - Sistema paramétrico
# ═══════════════════════════════════════════════════════════════

output_32 = widgets.Output()

def plot_parametrico(lam):
    with output_32:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

        # Panel izquierdo: gráfica
        ax = axes[0]
        x = np.linspace(-10, 10, 1000)

        y1 = (3 - x) / 2
        ax.plot(x, y1, 'b-', linewidth=2.5, label=f'Ec.1: x₁ + 2x₂ = 3')

        if abs(lam) > 0.001:
            y2 = (6 - 2*x) / lam
            mask = np.abs(y2) < 12
            ax.plot(x[mask], y2[mask], 'r-', linewidth=2.5, label=f'Ec.2: 2x₁ + {lam:.1f}x₂ = 6')
        else:
            ax.axvline(x=3, color='r', linewidth=2.5, label=f'Ec.2: 2x₁ = 6')

        det = lam - 4  # det = 1*lam - 2*2

        if abs(det) > 0.01:
            sol_x = (3*lam - 12) / det
            sol_y = (6 - 6) / det if abs(det) > 0.01 else 0
            sol_y = (1*6 - 2*3) / det  # = 0/det = 0
            sol_x = (3*lam - 2*6) / det
            sol_y = (1*6 - 2*3) / det
            ax.plot(sol_x, sol_y, 'go', markersize=15, zorder=5,
                    markeredgecolor='black', markeredgewidth=2)
            tipo = f'Solución única: ({sol_x:.2f}, {sol_y:.2f})'
            color_t = 'green'
        else:
            c_check = 1*6 - 2*3  # = 0
            if abs(c_check) < 0.01:
                tipo = 'Infinitas soluciones'
                color_t = 'blue'
            else:
                tipo = 'Sin solución'
                color_t = 'red'

        ax.set_xlim(-8, 8)
        ax.set_ylim(-8, 8)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='k', linewidth=0.5)
        ax.axvline(x=0, color='k', linewidth=0.5)
        ax.set_title(f'λ = {lam:.1f} → {tipo}', fontsize=12, color=color_t, fontweight='bold')
        ax.legend(fontsize=9)
        ax.set_xlabel('x₁')
        ax.set_ylabel('x₂')

        # Panel derecho: forma escalonada
        ax2 = axes[1]
        ax2.axis('off')

        A_aug = np.array([[1, 2, 3], [2, lam, 6]], dtype=float)
        A_esc = A_aug.copy()
        A_esc[1] = A_esc[1] - 2 * A_esc[0]

        texto = "Matriz aumentada:\n"
        texto += f"┌ {1:6.1f}  {2:6.1f} │ {3:6.1f} ┐\n"
        texto += f"└ {2:6.1f}  {lam:6.1f} │ {6:6.1f} ┘\n\n"
        texto += "Operación: F₂ → F₂ - 2·F₁\n\n"
        texto += "Forma escalonada:\n"
        texto += f"┌ {A_esc[0,0]:6.1f}  {A_esc[0,1]:6.1f} │ {A_esc[0,2]:6.1f} ┐\n"
        texto += f"└ {A_esc[1,0]:6.1f}  {A_esc[1,1]:6.1f} │ {A_esc[1,2]:6.1f} ┘\n\n"
        texto += f"det = λ - 4 = {lam:.1f} - 4 = {det:.1f}\n\n"

        if abs(det) > 0.01:
            texto += f"→ det ≠ 0 → SOLUCIÓN ÚNICA"
        else:
            texto += f"→ det = 0 → ¿Infinitas o ninguna?\n"
            texto += f"  Última fila: [0  0 | {A_esc[1,2]:.1f}]\n"
            if abs(A_esc[1,2]) < 0.01:
                texto += f"  0 = 0 ✓ → INFINITAS SOLUCIONES"
            else:
                texto += f"  0 = {A_esc[1,2]:.1f} ✗ → SIN SOLUCIÓN"

        ax2.text(0.05, 0.95, texto, transform=ax2.transAxes, fontsize=12,
                 verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

        plt.tight_layout()
        plt.show()

s_lam = widgets.FloatSlider(value=3.0, min=-2, max=8, step=0.1, description='λ:',
                              style={'description_width': '30px'},
                              layout=widgets.Layout(width='70%'))

ui4 = widgets.VBox([
    widgets.HTML('<h4>🎛️ Ajuste el hiperparámetro λ y observe cómo cambia la solución:</h4>'),
    s_lam
])

out4 = widgets.interactive_output(plot_parametrico, {'lam': s_lam})
display(ui4, output_32)
plot_parametrico(s_lam.value)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PREGUNTAS 3.2 — Escriba sus respuestas abajo
# ═══════════════════════════════════════════════════════════════

caja_respuesta("P16",
    "¿Para qué valor exacto de λ el sistema deja de tener solución única? "
    "¿Cómo lo determina a partir del determinante de la matriz de coeficientes?", 3)

caja_respuesta("P17",
    "Cuando λ = 4, ¿el sistema tiene infinitas soluciones o ninguna solución? "
    "Justifique observando la última fila de la forma escalonada.", 4)

caja_respuesta("P18",
    "En el contexto de MiniGPT: si λ es la tasa de aprendizaje y el sistema tiene infinitas "
    "soluciones, ¿qué significa esto para los pesos del modelo? ¿Y si no tiene solución?", 4)

---
## Parte 4: Actividad Integradora (~25 min)

### El desafío final de MiniGPT

MiniGPT necesita procesar un vocabulario de 4 palabras: **{"hola", "mundo", "adiós", "cielo"}**. Los embeddings propuestos en $\mathbb{R}^3$ son:

| Palabra | Embedding |
|---------|-----------|
| "hola"  | $\mathbf{v}_1 = (1, 0, 2)$ |
| "mundo" | $\mathbf{v}_2 = (0, 1, 1)$ |
| "adiós" | $\mathbf{v}_3 = (1, 1, 3)$ |
| "cielo" | $\mathbf{v}_4 = (2, -1, 3)$ |

---

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ACTIVIDAD 4.1 - Análisis de los embeddings
# ═══════════════════════════════════════════════════════════════

output_41 = widgets.Output()

v1 = np.array([1, 0, 2])
v2 = np.array([0, 1, 1])
v3 = np.array([1, 1, 3])
v4 = np.array([2, -1, 3])

def analizar_subconjunto(seleccion):
    with output_41:
        clear_output(wait=True)
        nombres = {'v₁ "hola"': v1, 'v₂ "mundo"': v2, 'v₃ "adiós"': v3, 'v₄ "cielo"': v4}

        if len(seleccion) == 0:
            print("Seleccione al menos un vector.")
            return

        vecs = [nombres[s] for s in seleccion]
        labels = list(seleccion)

        A = np.array(vecs).T  # Cada vector como columna
        rango = np.linalg.matrix_rank(A)
        n_vecs = len(vecs)

        print("=" * 55)
        print("  ANÁLISIS DEL SUBCONJUNTO DE EMBEDDINGS")
        print("=" * 55)
        print()
        print("Vectores seleccionados:")
        for l, v in zip(labels, vecs):
            print(f"  {l} = {v}")

        print(f"\nMatriz formada (vectores como columnas):")
        for fila in A:
            print(f"  {fila}")

        print(f"\nNúmero de vectores: {n_vecs}")
        print(f"Rango de la matriz:  {rango}")
        print(f"Dimensión del espacio: 3 (R³)")

        print()
        if rango == n_vecs:
            print("✅ Los vectores son LINEALMENTE INDEPENDIENTES")
            if n_vecs == 3:
                print("✅ Además, forman una BASE de R³")
                print("   → MiniGPT puede representar cualquier significado en R³")
            elif n_vecs < 3:
                print(f"⚠️ Pero solo generan un subespacio de dimensión {rango}")
                print(f"   → NO son suficientes para ser base de R³ (faltan {3-rango} vector(es))")
            else:
                print("⚠️ Hay más vectores que la dimensión del espacio")
                print("   → NO pueden ser base (una base de R³ tiene exactamente 3 vectores)")
        else:
            print("❌ Los vectores son LINEALMENTE DEPENDIENTES")
            print(f"   → {n_vecs - rango} vector(es) son combinación lineal de los otros")
            if n_vecs <= 3:
                print("   → NO forman base de R³")

        # Visualización 3D
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')

        colores = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']
        for idx, (l, v) in enumerate(zip(labels, vecs)):
            ax.quiver(0, 0, 0, v[0], v[1], v[2], color=colores[idx % 4],
                      linewidth=2.5, arrow_length_ratio=0.1, label=l)

        ax.set_xlim([-3, 3])
        ax.set_ylim([-3, 3])
        ax.set_zlim([-1, 4])
        ax.set_xlabel('Componente 1')
        ax.set_ylabel('Componente 2')
        ax.set_zlabel('Componente 3')
        ax.legend(loc='upper left', fontsize=8)

        estado = 'LI' if rango == n_vecs else 'LD'
        base = ' ✓ Base de R³' if (rango == n_vecs == 3) else ''
        ax.set_title(f'Embeddings en R³ — {estado}{base}', fontsize=12)

        plt.tight_layout()
        plt.show()

selector = widgets.SelectMultiple(
    options=['v₁ "hola"', 'v₂ "mundo"', 'v₃ "adiós"', 'v₄ "cielo"'],
    value=['v₁ "hola"', 'v₂ "mundo"', 'v₃ "adiós"'],
    description='Vectores:',
    style={'description_width': '70px'},
    layout=widgets.Layout(height='110px', width='40%')
)

btn_analizar = widgets.Button(description='🔍 Analizar subconjunto', button_style='primary',
                               layout=widgets.Layout(width='250px'))

def on_analizar(b):
    analizar_subconjunto(list(selector.value))

btn_analizar.on_click(on_analizar)

ui5 = widgets.VBox([
    widgets.HTML('<h4>🎛️ Seleccione un subconjunto de vectores (Ctrl+Click para múltiples):</h4>'),
    selector, btn_analizar
])

display(ui5, output_41)
analizar_subconjunto(list(selector.value))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PREGUNTAS 4.1 — Escriba sus respuestas abajo
# ═══════════════════════════════════════════════════════════════

caja_respuesta("P19",
    "Seleccione {v₁, v₂, v₃}. ¿Son linealmente independientes? "
    "¿Forman base de R³? Observe que v₃ = v₁ + v₂. ¿Qué implica esto?", 4)

caja_respuesta("P20",
    "Ahora seleccione {v₁, v₂, v₄}. ¿Son linealmente independientes? ¿Forman base de R³? "
    "Compare con el resultado anterior.", 4)

caja_respuesta("P21",
    "De los 4 vectores dados, ¿cuál es el número máximo de vectores linealmente independientes "
    "que puede seleccionar? ¿Puede encontrar un subconjunto que forme base de R³? Indique cuál.", 4)

### Actividad 4.2: Resolviendo el sistema de pesos de MiniGPT

Finalmente, para que MiniGPT aprenda a predecir la siguiente palabra, se debe resolver el sistema que surge de los datos de entrenamiento. Los embeddings base seleccionados son $\mathbf{v}_1$, $\mathbf{v}_2$ y $\mathbf{v}_4$. El sistema para los pesos de atención $w_1, w_2, w_3$ es:

$$\begin{cases} w_1 + \phantom{0}0w_2 + 2w_3 = 5 \\ 0w_1 + \phantom{0}w_2 + w_3 = 3 \\ 2w_1 - w_2 + 3w_3 = 8 \end{cases}$$

Use la herramienta de Gauss para resolver este sistema.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ACTIVIDAD 4.2 - Sistema final de pesos
# ═══════════════════════════════════════════════════════════════

print("Sistema de ecuaciones para los pesos de atención de MiniGPT:")
print("   w₁ + 0w₂ + 2w₃ = 5")
print("  0w₁ +  w₂ +  w₃ = 3")
print("  2w₁ -  w₂ + 3w₃ = 8")
print()
gauss_final = GaussInteractivo([[1, 0, 2, 5],
                                  [0, 1, 1, 3],
                                  [2, -1, 3, 8]], ['w₁', 'w₂', 'w₃'])

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PREGUNTAS FINALES 4.2 — Escriba sus respuestas abajo
# ═══════════════════════════════════════════════════════════════

caja_respuesta("P22",
    "Reduzca la matriz a forma escalonada. Liste las operaciones que realizó y escriba "
    "la forma escalonada resultante.", 5)

caja_respuesta("P23",
    "¿Cuántas soluciones tiene este sistema? ¿Es consistente o inconsistente? "
    "Si tiene solución, encuéntrela usando sustitución hacia atrás a partir de la forma escalonada. "
    "Indique los valores de w₁, w₂ y w₃.", 5)

caja_respuesta("P24",
    "Considere el sistema completo del entrenamiento de MiniGPT. Si la tercera ecuación "
    "se cambiara a 2w₁ − w₂ + 3w₃ = 10, ¿seguiría teniendo solución? "
    "¿Qué tipo? Justifique sin necesidad de resolver, usando lo que sabe sobre "
    "la forma escalonada y la consistencia de sistemas.", 4)

---
## 📝 Resumen de conceptos evaluados

| Concepto | Actividad |
|----------|-----------|
| Base y dimensión | 1.1, 1.2, 4.1 |
| Independencia lineal | 1.1, 4.1 |
| Combinaciones lineales | 1.2 |
| Operaciones elementales | 2.1, 2.2, 4.2 |
| Reducción de Gauss | 2.1, 2.2, 4.2 |
| Tipos de solución | 3.1, 3.2, 4.2 |
| Forma escalonada | 2.1, 2.2, 3.2, 4.2 |

---

### ⚠️ Antes de entregar

1. Verifique que **TODAS** las cajas de respuesta (P1 a P24) estén completas.
2. Guarde el cuaderno: `Ctrl+S` (Windows/Linux) o `Cmd+S` (Mac).
3. Suba el archivo `.ipynb` a la plataforma.

---
*Actividad Computacional 2 — Álgebra Lineal — 2026-01*

In [ ]:
# ═══════════════════════════════════════════════════════════════
# VERIFICACIÓN: Ejecute esta celda para ver un resumen de sus respuestas
# ═══════════════════════════════════════════════════════════════

print("=" * 60)
print("  RESUMEN DE RESPUESTAS REGISTRADAS")
print("=" * 60)

total = 24
respondidas = 0
for i in range(1, total + 1):
    pid = f"P{i}"
    r = respuestas.get(pid, "")
    estado = "✅" if len(r.strip()) > 0 else "⬜"
    if len(r.strip()) > 0:
        respondidas += 1
    preview = r.strip()[:60] + "..." if len(r.strip()) > 60 else r.strip()
    if preview:
        print(f"  {estado} {pid}: {preview}")
    else:
        print(f"  {estado} {pid}: (sin respuesta)")

print()
print(f"  Progreso: {respondidas}/{total} preguntas respondidas.")
if respondidas == total:
    print("  🎉 ¡Todas las preguntas están respondidas! No olvide guardar.")
else:
    print(f"  ⚠️ Faltan {total - respondidas} respuestas por completar.")